<!-- dads-lab-header -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M03/M03_Lab1_Prompting_Techniques.ipynb)

![M03 Lab1 Prompting Techniques](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M03/assets/images/M03_Lab1_Prompting_Techniques_banner.png)

In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai, setup_gemini) once per Colab runtime. The same OPENAI_API_KEY
# / GEMINI_API_KEY Colab secrets are used across every DADS 5250 lab — set
# them once in the 🔑 sidebar and they're picked up automatically.
import os
import importlib.util
if importlib.util.find_spec("dads5250") is None:
    !pip install -q dads5250==0.2.0

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    setup_gemini,
    DEFAULT_CHAT_MODEL,   # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,   # newest mini model that supports temperature
    DEFAULT_EMBED_MODEL,  # current embeddings default
    DEFAULT_GEMINI_MODEL, # tracks the latest stable flash
)

lab_pill('M03 Lab 1 — Prompting Techniques')            # sticky banner so you always see which lab you're in


## API check

Confirm the API connection before we start. Your key is read from a Colab Secret, an environment variable, or a hidden prompt if neither is set.

In [ ]:
# === API check: confirm the connection and show the model(s) this lab uses ===
client = setup_openai()        # loads OPENAI_API_KEY + verifies it works

pp({
    "OpenAI":      "connected",
    "chat model":  DEFAULT_CHAT_MODEL,
    "mini model":  DEFAULT_MINI_MODEL,
}, title="API check")

# Part 1 — Fundamentals: Prompting with the API

Before strategies, the basics: how you talk to the model through **roles**. Every request is a list of messages, each with a role that shapes the conversation.

- **System** — sets the behavior and tone ("You are a strict teacher").
- **User** — your input or question.
- **Assistant** — the model's reply (you can also supply prior assistant turns to steer it).

In [ ]:
# ==========================================================
# 1. Roles: system + user (a basic call)
# ==========================================================
from openai import OpenAI
client = OpenAI()

import openai

response = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

pretty_print(response.choices[0].message.content, title="🤖 Model Response")

### Adding an assistant turn

You can include a prior **assistant** message to show the model the style of answer you want.

In [ ]:
# ==========================================================
# 2. Roles: adding an assistant turn
# ==========================================================
from openai import OpenAI
client = OpenAI()

# Demonstrating roles in OpenAI's API with the assistant role

# Define the conversation including a predefined assistant response
messages = [
    {"role": "system", "content": "You are a math tutor who explains problems step by step."},  # System role sets the behavior
    {"role": "user", "content": "Solve for x: 2x + 5 = 15"},  # User question
    {"role": "assistant", "content": "To solve for x: \n1. Subtract 5 from both sides: 2x = 10\n2. Divide both sides by 2: x = 5"}  # Predefined assistant response
]

# Send the conversation to the API
response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,  # Use the chosen model
    messages=messages  # Pass the conversation
)

# Print the assistant's response
pretty_print(response.choices[0].message.content, title="Assistant's Response:")

### Hands-on: complete the roles

Replace each `----` with the correct role (`system`, `user`, `assistant`).

In [ ]:
# ==========================================================
# 3. Hands-on: complete the roles
# ==========================================================
from openai import OpenAI
client = OpenAI()

# ++++ Hands-On: Complete the Roles ++++

# Define the conversation using the role structure
messages = [
    {"role": "----", "content": "You are a math tutor who explains concepts clearly and step by step."},  # Complete the system role
    {"role": "----", "content": "How do you calculate the arithmetic mean of a set of numbers?"},  # Complete the user role
    {"role": "----", "content": "To calculate the arithmetic mean:\n"
                                "1. Add all the numbers in the set.\n"
                                "2. Divide the sum by the number of numbers.\n\n"
                                "For example, for 10, 20, and 30:\n"
                                "Mean = (10 + 20 + 30) / 3 = 60 / 3 = 20."
                                }  # Optional predefined assistant response
]

# Uncomment the code below after completing the placeholders
# response = client.chat.completions.create(
#     model=DEFAULT_CHAT_MODEL,  # Specify the model
#     messages=messages  # Pass the completed conversation
# )
# pretty_print(response.choices[0].message.content, title="Assistant's Response:")

### Exercise 1 — Q&A with system + user

Make the assistant behave like a science teacher, then ask it a science question.

In [ ]:
# ==========================================================
# 4. Exercise 1: your system + user prompt
# ==========================================================
from openai import OpenAI
client = OpenAI()

# 🧠 Define your own roles and prompt below
messages = [
    {"role": "------", "content": "You are a [insert role/personality, e.g., 'science teacher', 'debate coach']."},
    {"role": "------", "content": "Ask your question here, e.g., 'Explain how photosynthesis works.'"}
]

# 🧠 Call the OpenAI API with your customized prompt
response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,
    messages=messages
)

# 🧠 Print the model's response
pretty_print(response.choices[0].message.content, title="Assistant's Response:")

### Exercise 2 — your own prompt

Any role, any question. Be creative.

In [ ]:
# ==========================================================
# 5. Exercise 2: a prompt of your choice
# ==========================================================
from openai import OpenAI
client = OpenAI()

# Replace with your own prompt idea!
messages = [
    {"role": "system", "content": "You are a ____."},
    {"role": "user", "content": "____?"}
]

response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,
    messages=messages
)

pretty_print(response.choices[0].message.content, title="Assistant's Response:")

# Part 2 — Prompting Strategies

**Why strategies matter.** Vague prompts get vague answers. The way you frame a task, and whether you give examples or ask for reasoning, changes the quality a lot. We go from the basic shot types to advanced reasoning techniques.

**Basic types:**
- **Zero-shot** — just the task, no examples. Use when the task is simple and familiar.
- **One-shot** — one worked example, then the task.
- **Few-shot** — several examples to lock in a pattern or format.

In [ ]:
# ==========================================================
# 6. Zero-shot: a hidden formula sequence
# ==========================================================
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Zero-Shot Test: Hidden Formula Sequence
# ==========================

hard_sequence_prompt_zero = (
    "The sequence is: 3, 12, 27, 48, 75, ___. What’s next?"
)

response_zero_hard = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[{"role": "user", "content": hard_sequence_prompt_zero}],
    temperature=0
)

print("🔹 LLM Response (Zero-Shot - Hard Sequence):\n")
pretty_print(response_zero_hard.choices[0].message.content.strip(), title="🤖 Model Response")

### Zero-shot vs one-shot

One clear example can flip a wrong answer into a right one.

In [ ]:
# ==========================================================
# 7. Zero-shot vs one-shot on the same sequence
# ==========================================================
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Zero-Shot vs One-Shot Comparison: Alternating Pattern Sequence (Correct One-Shot)
# ==========================

model_name = DEFAULT_MINI_MODEL

# Zero-Shot Prompt (No Example)
zero_shot_prompt = (
    "The sequence is: 1, 4, 2, 9, 3, 16, 4, ___. What number should replace the blank?"
)

# One-Shot Prompt (One Example + New Question)
one_shot_prompt = (
    "Example:\n"
    "The sequence is: 1, 1, 2, 4, 3, 9, ___. What’s next?\n"
    "Answer: 4.\n\n"
    "Now solve this one:\n"
    "The sequence is: 1, 4, 2, 9, 3, 16, 4, ___. What number should replace the blank?"
)

# Run Zero-Shot
response_zero = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0
)

# Run One-Shot
response_one = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": one_shot_prompt}],
    temperature=0
)

# Display Results
print("🔹 Zero-Shot Response:\n" + "-"*40)
pretty_print(response_zero.choices[0].message.content.strip(), title="🤖 Model Response")

print("\n\n🔹 One-Shot Response:\n" + "-"*40)
pretty_print(response_one.choices[0].message.content.strip(), title="🤖 Model Response")

### Few-shot

Several examples to teach a harder, multi-rule pattern.

In [ ]:
# ==========================================================
# 8. Few-shot: an ultra-hard pattern
# ==========================================================
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Few-Shot Prompting Example: Ultra-Hard Pattern (3 Hidden Rules)
# ==========================

model_name = DEFAULT_CHAT_MODEL  # Best for complex reasoning

# Few-Shot Prompt with 2 Examples
few_shot_prompt = (
    "Example 1:\n"
    "The sequence is: 1, 1, 2, 4, 3, 9, ___. What’s next?\n"
    "Answer: 4.\n\n"
    "Example 2:\n"
    "The sequence is: 1, 1, 2, 4, 4, 9, 7, 16, ___. What’s next?\n"
    "Answer: 11.\n\n"
    "Now try this one:\n"
    "The sequence is: 1, 1, 2, 4, 4, 9, 7, 16, 11, ___, 16, 36. What number should replace the blank?"
)

# Run Few-Shot Prompt
response_few = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0
)

# Display Result
print("🔹 Few-Shot Prompting (Two Examples Provided):")
print("-" * 40)
pretty_print(response_few.choices[0].message.content.strip(), title="🤖 Model Response")

## Advanced techniques

### Chain-of-Thought (CoT)

CoT asks the model to **show its intermediate reasoning** before answering, which improves multi-step problems. Reference: [Wei et al., 2022](https://arxiv.org/abs/2201.11903).

In [ ]:
# ==========================================================
# 9. Chain-of-Thought: force step-by-step reasoning
# ==========================================================
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Chain-of-Thought Demonstration: Make 110 with Five 5's
# ==========================

model_name = DEFAULT_CHAT_MODEL

# Zero-Shot Prompt (No Reasoning Encouraged)
zero_shot_prompt = (
    "Use exactly five 5’s and only four operations (+, -, *, /) and parentheses to make 110."
)

# Chain-of-Thought Prompt (Encourages Step-by-Step Reasoning)
cot_prompt = (
    "Let's solve this step by step.\n"
    "We need to use exactly five 5’s and only four operations (+, -, *, /) and parentheses to make 110.\n"
    "Step 1: Think about how we can combine the 5's to form larger numbers (e.g., 55).\n"
    "Step 2: Try to combine them logically to reach 110.\n"
    "Now, provide the final equation and the answer."
)

# Run Zero-Shot
response_zero = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0
)

# Run Chain-of-Thought
response_cot = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": cot_prompt}],
    temperature=0
)

# Display Results
print("🔹 Zero-Shot Response (No Reasoning Encouraged):\n" + "-" * 50)
pretty_print(response_zero.choices[0].message.content.strip(), title="🤖 Model Response")

print("\n🔹 Chain-of-Thought Response (Reasoning Encouraged):\n" + "-" * 50)
pretty_print(response_cot.choices[0].message.content.strip(), title="🤖 Model Response")

> **Pause and think.** Run the two prompts above. Did zero-shot get it? Did the step-by-step version reason more clearly? Jot what you notice.

**Your notes** *(double-click to edit)*

- Zero-shot result: 
- Chain-of-Thought result: 
- What changed: 

### Self-consistency

CoT can still land on a wrong path. **Self-consistency** samples several reasoning paths (nonzero temperature) and takes the most common answer. Reference: [Wang et al., 2022](https://arxiv.org/abs/2203.11171).

In [ ]:
# ==========================================================
# 10. Self-consistency: sample many paths, take the majority
# ==========================================================
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Comparing Chain-of-Thought vs. Self-Consistency Prompting
# ==========================

model_name = DEFAULT_CHAT_MODEL  # Using GPT-4 for better reasoning

# Define the problem prompt
problem_prompt = (
    "If a train travels at 60 miles per hour and leaves at 2 PM, and another train leaves "
    "the same station at 3 PM traveling at 90 miles per hour, when will the second train catch up to the first?"
)

# Chain-of-Thought Prompt (Standard)
cot_prompt = (
    "Let's solve this step by step.\n"
    + problem_prompt
)

# Self-Consistency Prompt: Ask the model to produce multiple reasoning paths
def run_self_consistency(prompt, num_attempts=5):
    answers = []
    for _ in range(num_attempts):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7  # Add randomness to explore different reasoning paths
        )
        answer = response.choices[0].message.content.strip()
        answers.append(answer)
    return answers

# Run Chain-of-Thought (Single Attempt)
response_cot = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": cot_prompt}],
    temperature=0
)
cot_answer = response_cot.choices[0].message.content.strip()

# Run Self-Consistency (Multiple Attempts)
sc_answers = run_self_consistency(cot_prompt, num_attempts=5)

# Simple Majority Vote to Find Most Consistent Answer
from collections import Counter
most_common_answer = Counter(sc_answers).most_common(1)[0]

# Display Results
print("🔹 Chain-of-Thought Response (Single Attempt):\n" + "-" * 50)
print(cot_answer)

print("\n🔹 Self-Consistency Responses (Multiple Attempts):\n" + "-" * 50)
for idx, ans in enumerate(sc_answers, 1):
    print(f"Attempt {idx}: {ans}")

print("\n🔹 Final Self-Consistency Selected Answer:\n" + "-" * 50)
print(f"Most Common Answer: {most_common_answer[0]}\nAppeared {most_common_answer[1]} times.")

### Even more strategies

- **Tree-of-Thought (ToT)** — explore multiple reasoning branches like a decision tree.
- **ReAct** — interleave reasoning with actions (tool/API calls).
- **Reflexion** — the model critiques and revises its own answer.

We touch these more in the agents modules.

In [ ]:
# ==========================================================
# 11. Hands-on: try different strategies and models
# ==========================================================
# ==========================
# ✋ Hands-On Code: Try Different Prompting Strategies and Models
# ==========================

# 📝 Instructions:
# - Change 'model_name' to try different models (e.g., DEFAULT_MINI_MODEL, DEFAULT_CHAT_MODEL, "gpt-o3").
# - Adjust 'temperature' to test how creativity affects reasoning.
# - Try Self-Consistency by sampling multiple outputs and comparing answers.
# - Optionally, explore Tree-of-Thought and ReAct patterns by modifying prompts.
# ✅ Your Experiment Starts Here 👇

## Wrap-up

You moved from the basics (roles) to strategy: zero-, one-, and few-shot, then Chain-of-Thought and self-consistency. The through-line: **the more structure and reasoning you invite, the better the model does on hard tasks** — at some cost in tokens and time. Match the technique to the task.